# 03 - SCD Type 2 – Vendor Contracts

## Procurement Analytics Pipeline

This notebook implements **Slowly Changing Dimension (SCD) Type 2** for vendor contracts.

### Objective

Track historical changes in vendor contracts such as:

* Negotiated price changes
* Contract validity changes
* Multiple versions of the same vendor-item contract

The SCD Type 2 table preserves historical records while maintaining a single current active record.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, row_number
from pyspark.sql.window import Window
import os

## Initialize Spark Session

Create a Spark session for SCD processing.


In [2]:
spark = SparkSession.builder \
    .appName('SCD Type 2') \
    .master('local[*]') \
    .getOrCreate()

print('Spark Started')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Started


## Load Silver Contracts Dataset

Load the curated contracts dataset prepared in the Silver layer.


In [3]:
contracts_df = spark.read.option('header', True).csv('output/silver/contracts_silver.csv')

contracts_df.printSchema()
contracts_df.show(5, truncate=False)

root
 |-- contract_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- negotiated_price: string (nullable = true)
 |-- valid_from: string (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- effective_start_date: string (nullable = true)
 |-- effective_end_date: string (nullable = true)
 |-- is_current: string (nullable = true)

+-----------+---------+-------------+----------------+----------+--------------------------+--------------------+------------------+----------+
|contract_id|vendor_id|item_name    |negotiated_price|valid_from|ingestion_timestamp       |effective_start_date|effective_end_date|is_current|
+-----------+---------+-------------+----------------+----------+--------------------------+--------------------+------------------+----------+
|C000001    |V04273   |Solutions    |1745.93         |2024-04-25|2026-08-12 21:45:41.677177|2024-04-25          |NULL              |True      |
|C000002

## Standardize Data Types

Convert price and timestamp columns to appropriate data types.


In [4]:
contracts_df = contracts_df \
    .withColumn('negotiated_price', col('negotiated_price').cast('double')) \
    .withColumn('valid_from', col('valid_from').cast('timestamp'))

contracts_df.printSchema()

root
 |-- contract_id: string (nullable = true)
 |-- vendor_id: string (nullable = true)
 |-- item_name: string (nullable = true)
 |-- negotiated_price: double (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- ingestion_timestamp: string (nullable = true)
 |-- effective_start_date: string (nullable = true)
 |-- effective_end_date: string (nullable = true)
 |-- is_current: string (nullable = true)



## Identify Contract Versions

For each combination of **vendor_id** and **item_name**, contracts are ordered by validity date to identify historical versions.


In [5]:
window_spec = Window.partitionBy(
    'vendor_id',
    'item_name'
).orderBy('valid_from')

contracts_versioned = contracts_df.withColumn(
    'version_number',
    row_number().over(window_spec)
)

contracts_versioned.select(
    'vendor_id',
    'item_name',
    'valid_from',
    'version_number'
).show(10, truncate=False)

+---------+------------+-------------------+--------------+
|vendor_id|item_name   |valid_from         |version_number|
+---------+------------+-------------------+--------------+
|V00003   |Services    |2023-07-13 00:00:00|1             |
|V00004   |Bandwidth   |2024-02-07 00:00:00|1             |
|V00004   |Roi         |2024-12-06 00:00:00|1             |
|V00010   |Markets     |2023-05-10 00:00:00|1             |
|V00012   |Users       |2023-07-17 00:00:00|1             |
|V00013   |Models      |2023-05-17 00:00:00|1             |
|V00017   |Action-items|2025-12-12 00:00:00|1             |
|V00017   |Platforms   |2023-05-02 00:00:00|1             |
|V00020   |Schemas     |2024-04-03 00:00:00|1             |
|V00021   |Users       |2025-06-25 00:00:00|1             |
+---------+------------+-------------------+--------------+
only showing top 10 rows


## Create SCD Type 2 Columns

Add effective start date, effective end date, and current flag columns.


In [6]:
from pyspark.sql.functions import lead, when

contracts_scd = contracts_versioned.withColumn(
    'effective_start_date',
    col('valid_from')
)

contracts_scd = contracts_scd.withColumn(
    'next_valid_from',
    lead('valid_from').over(window_spec)
)

contracts_scd = contracts_scd.withColumn(
    'effective_end_date',
    col('next_valid_from')
)

contracts_scd = contracts_scd.withColumn(
    'is_current',
    when(col('next_valid_from').isNull(), lit(True)).otherwise(lit(False))
).drop('next_valid_from')

contracts_scd.show(10, truncate=False)

+-----------+---------+------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|contract_id|vendor_id|item_name   |negotiated_price|valid_from         |ingestion_timestamp       |effective_start_date|effective_end_date|is_current|version_number|
+-----------+---------+------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|C000720    |V00003   |Services    |1893.14         |2023-07-13 00:00:00|2026-08-12 21:45:41.677177|2023-07-13 00:00:00 |NULL              |true      |1             |
|C001395    |V00004   |Bandwidth   |230.96          |2024-02-07 00:00:00|2026-08-12 21:45:41.677177|2024-02-07 00:00:00 |NULL              |true      |1             |
|C003062    |V00004   |Roi         |2813.17         |2024-12-06 00:00:00|2026-08-12 21:45:41.677177|2024-12-06 00:00:00 |NULL              |true      |1             

## Example History for a Single Vendor

Display all contract versions for one vendor-item combination.


In [7]:
sample_vendor = contracts_scd.select('vendor_id').first()[0]

contracts_scd.filter(
    col('vendor_id') == sample_vendor
).orderBy('item_name', 'effective_start_date').show(truncate=False)

+-----------+---------+---------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|contract_id|vendor_id|item_name|negotiated_price|valid_from         |ingestion_timestamp       |effective_start_date|effective_end_date|is_current|version_number|
+-----------+---------+---------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|C000001    |V04273   |Solutions|1745.93         |2024-04-25 00:00:00|2026-08-12 21:45:41.677177|2024-04-25 00:00:00 |NULL              |true      |1             |
+-----------+---------+---------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+



## Current Active Contracts

Filter only the latest active contract records.


In [8]:
current_contracts = contracts_scd.filter(
    col('is_current') == True
)

print('Current active contracts:', current_contracts.count())
current_contracts.show(10, truncate=False)

Current active contracts: 4897
+-----------+---------+------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|contract_id|vendor_id|item_name   |negotiated_price|valid_from         |ingestion_timestamp       |effective_start_date|effective_end_date|is_current|version_number|
+-----------+---------+------------+----------------+-------------------+--------------------------+--------------------+------------------+----------+--------------+
|C000720    |V00003   |Services    |1893.14         |2023-07-13 00:00:00|2026-08-12 21:45:41.677177|2023-07-13 00:00:00 |NULL              |true      |1             |
|C001395    |V00004   |Bandwidth   |230.96          |2024-02-07 00:00:00|2026-08-12 21:45:41.677177|2024-02-07 00:00:00 |NULL              |true      |1             |
|C003062    |V00004   |Roi         |2813.17         |2024-12-06 00:00:00|2026-08-12 21:45:41.677177|2024-12-06 00:00:00 |NULL         

## Historical Contract Records

Filter historical contract versions.


In [9]:
historical_contracts = contracts_scd.filter(
    col('is_current') == False
)

print('Historical contract records:', historical_contracts.count())
historical_contracts.show(10, truncate=False)

Historical contract records: 103
+-----------+---------+-------------+----------------+-------------------+--------------------------+--------------------+-------------------+----------+--------------+
|contract_id|vendor_id|item_name    |negotiated_price|valid_from         |ingestion_timestamp       |effective_start_date|effective_end_date |is_current|version_number|
+-----------+---------+-------------+----------------+-------------------+--------------------------+--------------------+-------------------+----------+--------------+
|C003278    |V00079   |Solutions    |4275.78         |2024-02-12 00:00:00|2026-08-12 21:45:41.677177|2024-02-12 00:00:00 |2026-01-04 00:00:00|false     |1             |
|C004640    |V00256   |Paradigms    |1675.62         |2023-10-23 00:00:00|2026-08-12 21:45:41.677177|2023-10-23 00:00:00 |2023-12-13 00:00:00|false     |1             |
|C001826    |V00322   |Platforms    |1193.92         |2024-07-04 00:00:00|2026-08-12 21:45:41.677177|2024-07-04 00:00:00 |

## Save SCD Type 2 Output

Save the complete SCD history table and the current active contracts table.


In [10]:
os.makedirs('output/silver/scd', exist_ok=True)

contracts_scd.toPandas().to_csv(
    'output/silver/scd/contracts_scd_history.csv',
    index=False
)

current_contracts.toPandas().to_csv(
    'output/silver/scd/current_contracts.csv',
    index=False
)

print('SCD files saved successfully')

d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pand

SCD files saved successfully


d:\Celebal-Data-Engineering-Internship-Journey\venv\Lib\site-packages\pyspark\sql\pandas\types.py:778: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


## Verify Saved Files

Confirm that SCD output files were created successfully.


In [11]:
print(os.listdir('output/silver/scd'))

['contracts_scd_history.csv', 'current_contracts.csv']


## SCD Type 2 Summary

The following columns were created:

| Column               | Purpose                     |
| -------------------- | --------------------------- |
| version_number       | Sequential contract version |
| effective_start_date | Start date of the version   |
| effective_end_date   | End date of the version     |
| is_current           | Indicates active contract   |

This structure preserves complete contract history while enabling easy access to the current active contract.


# Conclusion

SCD Type 2 was successfully implemented for vendor contracts. The solution preserves historical contract versions, tracks contract changes over time, and maintains a current active contract table. The generated SCD history will be used in the Gold layer for vendor spend, price variance, and procurement analytics.
